# Демо 3. RAG на LangChain і оцінювання

У демо 2 ми написали RAG-пайплайн вручну. Тут збираємо той самий пайплайн із готових компонентів
[LangChain](https://python.langchain.com/), додаємо два способи покращити пошук — **гібридний пошук**
і **переранжування** — і, головне, **вимірюємо** якість системи на наборі питань із відомими відповідями.

Ретрив і генерацію оцінюємо окремо:
- **ретрив** — чи потрапляє фрагмент із потрібного документа до знайдених (hit rate@k, MRR);
- **генерація** — чи правильна відповідь, чи посилається вона на правильне джерело
  і чи відмовляється система відповідати, коли відповіді в документах немає.

_Для цього ноутбука потрібен GPU ._

In [1]:
# LangChain розбитий на пакети: core — базові абстракції, text-splitters — розбиття тексту,
# community — інтеграції (FAISS, BM25), huggingface — моделі з Hugging Face
!pip install -q "sentence-transformers>=5" pypdf langchain-core langchain-text-splitters langchain-community langchain-huggingface faiss-cpu rank_bm25

ERROR: pip's dependency resolver does not currently take into account all the packages that are installed. This behaviour is the source of the following dependency conflicts.
google-colab 1.0.0 requires requests==2.32.4, but you have requests 2.34.2 which is incompatible.


In [2]:
import re
import time
from pathlib import Path

import numpy as np
import pandas as pd
import requests
import torch
from langchain_community.retrievers import BM25Retriever
from langchain_community.vectorstores import FAISS
from langchain_community.vectorstores.utils import DistanceStrategy
from langchain_core.documents import Document
from langchain_core.output_parsers import StrOutputParser
from langchain_core.prompts import ChatPromptTemplate
from langchain_core.runnables import RunnableLambda, RunnablePassthrough
from langchain_huggingface import ChatHuggingFace, HuggingFaceEmbeddings, HuggingFacePipeline
from langchain_text_splitters import RecursiveCharacterTextSplitter
from pypdf import PdfReader
from sentence_transformers import CrossEncoder
from transformers import AutoModelForCausalLM, AutoTokenizer, pipeline

device = "cuda" if torch.cuda.is_available() else "cpu"
dtype = torch.float16 if device == "cuda" else torch.float32
print(f"Пристрій: {device}")

/tmp/ipykernel_676/1402898701.py:9: DeprecationWarning: `langchain-community` is being sunset and is no longer actively maintained. See https://github.com/langchain-ai/langchain-community/issues/674 for details and migration guidance toward standalone integration packages.
  from langchain_community.retrievers import BM25Retriever


Пристрій: cuda


## 1. Документи

Ті самі 14 документів, що в демо 1 і 2. LangChain має готові завантажувачі PDF, але колонтитули
й скани все одно доводиться обробляти самостійно, тому текст витягуємо так само, як раніше,
а потім загортаємо кожен документ в обʼєкт `Document`: текст плюс метадані.

In [3]:
# @title Завантаження й очищення документів (як у демо 1)
BASE_URL = "https://docs.ztu.edu.ua/"
DOCUMENTS = [
    ("275", "Положення про оцінювання результатів навчання (чинна версія 4.0)", {"mdocs-file": 275}),
    ("275-v3.1", "Положення про оцінювання результатів навчання (стара версія 3.1)",
     {"mdocs-version": "polozhennja pro ocinjuvannja znan yekts_nove.pdf-v3.1", "mdocs-file": 275, "mdocs-url": "false"}),
    ("277", "Положення про порядок та умови обрання здобувачами вищої освіти навчальних дисциплін", {"mdocs-file": 277}),
    ("291", "Положення про проведення практики здобувачів вищої освіти", {"mdocs-file": 291}),
    ("340", "Правила призначення академічних стипендій", {"mdocs-file": 340}),
    ("344", "Положення про стипендію ректора", {"mdocs-file": 344}),
    ("348", "Положення про користування гуртожитками", {"mdocs-file": 348}),
    ("1219", "Правила внутрішнього розпорядку в студентських гуртожитках", {"mdocs-file": 1219}),
    ("1199", "Кодекс академічної доброчесності", {"mdocs-file": 1199}),
    ("4389", "Положення про студентського омбудсмана", {"mdocs-file": 4389}),
    ("336", "Положення про старосту академічної групи", {"mdocs-file": 336}),
    ("3320", "ОПП F2 «Інженерія програмного забезпечення» (бакалавр), редакція 2025 року", {"mdocs-file": 3320}),
    ("3329", "ОПП 121 «Інженерія програмного забезпечення» (бакалавр), редакція 2024 року", {"mdocs-file": 3329}),
    ("283", "Положення про переведення здобувачів вищої освіти", {"mdocs-file": 283}),
]


PDF_DIR = Path("ztu_pdfs")
PDF_DIR.mkdir(exist_ok=True)


def fetch(params, attempts=3):
    # Сайт іноді обриває зʼєднання, тому пробуємо кілька разів
    for attempt in range(attempts):
        try:
            response = requests.get(BASE_URL, params=params, timeout=120)
            response.raise_for_status()
            return response.content
        except requests.RequestException:
            if attempt == attempts - 1:
                raise
            time.sleep(5)


def download_pdf(doc_id, params):
    # Зберігаємо PDF на диск, щоб під час повторного запуску не завантажувати їх знову
    path = PDF_DIR / f"{doc_id}.pdf"
    if not path.exists():
        path.write_bytes(fetch(params))
    return PdfReader(path)


HEADER_LINE = re.compile(
    r"^(Житомирська|політехніка|Міністерство освіти.*|Державний університет «Житомирська політехніка»|"
    r"Система управління якістю.*|П[\s\-–]*[CС]?[АA]?[ГG]?[\s\d.\-–]*|[\d.\-–\s/_]+|Випуск.*)$",
    re.IGNORECASE,
)
HEADER_LINES = 12


def page_text(page):
    lines = (page.extract_text() or "").split("\n")
    head = [line for line in lines[:HEADER_LINES] if not HEADER_LINE.match(line.strip())]
    return " ".join(head + lines[HEADER_LINES:])


MIN_CHARS_PER_PAGE = 100


def document_text(reader):
    return re.sub(r"\s+", " ", " ".join(page_text(page) for page in reader.pages)).strip()


def is_scan(text, reader):
    return len(text) < MIN_CHARS_PER_PAGE * len(reader.pages)


documents = {}
for doc_id, title, params in DOCUMENTS:
    reader = download_pdf(doc_id, params)
    text = document_text(reader)
    if is_scan(text, reader):
        print(f"{doc_id}: текстового шару немає, потрібне OCR — пропущено")
        continue
    documents[doc_id] = {"title": title, "text": text}
    print(f"{doc_id}: {len(reader.pages)} сторінок, {len(text)} символів — {title}")

275: 23 сторінок, 24944 символів — Положення про оцінювання результатів навчання (чинна версія 4.0)
275-v3.1: 13 сторінок, 10505 символів — Положення про оцінювання результатів навчання (стара версія 3.1)
277: 16 сторінок, 15945 символів — Положення про порядок та умови обрання здобувачами вищої освіти навчальних дисциплін
291: 17 сторінок, 14603 символів — Положення про проведення практики здобувачів вищої освіти
340: 30 сторінок, 35706 символів — Правила призначення академічних стипендій
344: 10 сторінок, 2455 символів — Положення про стипендію ректора
348: 22 сторінок, 25608 символів — Положення про користування гуртожитками
1219: 12 сторінок, 9159 символів — Правила внутрішнього розпорядку в студентських гуртожитках
1199: 37 сторінок, 56918 символів — Кодекс академічної доброчесності
4389: 18 сторінок, 20685 символів — Положення про студентського омбудсмана
336: 8 сторінок, 3560 символів — Положення про старосту академічної групи
3320: 16 сторінок, 31670 символів — ОПП F2 «Інженері

In [4]:
docs = [
    Document(page_content=doc["text"], metadata={"doc_id": doc_id, "title": doc["title"]})
    for doc_id, doc in documents.items()
]
print(docs[0].metadata, docs[0].page_content[:200])

{'doc_id': '275', 'title': 'Положення про оцінювання результатів навчання (чинна версія 4.0)'} ЗАТВЕРДЖЕНО Наказ Державного університету «Житомирська політехніка» 15 вересня 2025 р. № 327/од Ректор Віктор ЄВДОКИМОВ ПОЛОЖЕННЯ про оцінювання результатів навчання здобувачів вищої освіти у Державно


## 2. Розбиття на фрагменти

`RecursiveCharacterTextSplitter` пробує різати текст за першим роздільником зі списку; якщо шматок
усе ще задовгий, переходить до наступного роздільника. Наш список: кінець речення, крапка з комою,
пробіл і, в крайньому разі, будь-який символ. Метадані документа копіюються в кожен фрагмент.
Як і раніше, дописуємо на початок фрагмента назву документа.

In [5]:
splitter = RecursiveCharacterTextSplitter(
    chunk_size=1000, chunk_overlap=200, separators=[". ", "; ", " ", ""], keep_separator="end"
)
chunks = splitter.split_documents(docs)
for chunk in chunks:
    chunk.page_content = f"{chunk.metadata['title']}. {chunk.page_content}"

print(f"Фрагментів: {len(chunks)}")
print(chunks[100].metadata)
print(chunks[100].page_content)

Фрагментів: 385
{'doc_id': '340', 'title': 'Правила призначення академічних стипендій'}
Правила призначення академічних стипендій. 1.9. Студенти, аспіранти (докторанти), які реалізують право на академічну мобільність і зберігають статус здобувача вищої освіти (наукового ступеня) за денною формою навчання (навчаються з відривом від виробництва) у Житомирській політехніці, протягом строку навчання за програмою академічної мобільності в іншому навчальному закладі поза межами України отримують стипендію, призначену відповідно до цих Правил, у разі, коли умовами договору про навчання за програмою академічної мобільності, укладеного ними з Житомирською політехнікою, регулярна безповоротна фіксована фінансова підтримка у грошовій формі протягом всього строку навчання за програмою академічної мобільності в іншому навчальному закладі не передбачена або передбачена у розмірі, що у перерахунку на національну грошову одиницю на дату укладення договору про академічну мобільність є меншою за розмір 

## 3. Векторне сховище

`HuggingFaceEmbeddings` — обгортка LangChain над sentence-transformers. Для документів і запитів
задаємо різні параметри: запит кодується з промптом `query`, як у демо 1.
[FAISS](https://github.com/facebookresearch/faiss) — бібліотека векторного пошуку від Meta;
для нормалізованих векторів обираємо скалярний добуток як міру подібності.

In [6]:
EMBEDDING_MODEL_NAME = "Qwen/Qwen3-Embedding-0.6B"
embeddings = HuggingFaceEmbeddings(
    model_name=EMBEDDING_MODEL_NAME,
    model_kwargs={"device": device, "model_kwargs": {"dtype": dtype}},
    encode_kwargs={"normalize_embeddings": True, "batch_size": 16},
    query_encode_kwargs={"normalize_embeddings": True, "prompt_name": "query"},
)
vector_store = FAISS.from_documents(chunks, embeddings, distance_strategy=DistanceStrategy.MAX_INNER_PRODUCT)

for doc, score in vector_store.similarity_search_with_score("Коли обирають старосту на першому курсі?", k=3):
    print(f"{score:.2f}  [{doc.metadata['doc_id']}] {doc.page_content[:120]}…")

modules.json:   0%|          | 0.00/349 [00:00<?, ?B/s]

config_sentence_transformers.json:   0%|          | 0.00/215 [00:00<?, ?B/s]

README.md:   0%|          | 0.00/17.2k [00:00<?, ?B/s]

config.json:   0%|          | 0.00/727 [00:00<?, ?B/s]

model.safetensors: reconstructing file:   0%|          |  0.00B / 1.19GB            

model.safetensors: downloading bytes:           |  0.00B            

... (4 outputs left out)

tokenizer.json: reconstructing file:   0%|          |  0.00B / 11.4MB            

tokenizer.json: downloading bytes:           |  0.00B            

config.json:   0%|          | 0.00/313 [00:00<?, ?B/s]

0.60  [336] Положення про старосту академічної групи. У своїй роботі староста академічної групи співпрацює з куратором та ментором г…
0.53  [336] Положення про старосту академічної групи. До виборів обов’язки старости групи може виконувати один із студентів групи за…
0.50  [4389] Положення про студентського омбудсмана. 6.4. Студентський омбудсман взаємодіє з Молодіжною радою при Житомирській обласн…


## 4. Лексичний пошук BM25

BM25 — класичний алгоритм пошуку за збігом слів: фрагмент отримує тим більшу оцінку, чим більше в ньому
слів запиту і чим рідше ці слова трапляються в інших фрагментах. BM25 не розуміє синонімів і,
без додаткової обробки, різних форм того самого слова («гуртожиток» і «гуртожитку» для нього різні слова),
зате точно знаходить рідкісні слова, номери й коди.

In [7]:
CANDIDATES = 20

def tokenize(text):
    return re.findall(r"\w+", text.lower())

bm25 = BM25Retriever.from_documents(chunks, k=CANDIDATES, preprocess_func=tokenize)

def vector_search(query, k):
    return vector_store.similarity_search(query, k=k)

def bm25_search(query, k):
    return bm25.invoke(query)[:k]

def show(search, query, k=3):
    print(f"{search.__name__}: {query}")
    for doc in search(query, k):
        print(f"  [{doc.metadata['doc_id']}] {doc.page_content[:110]}…")
    print()

for query in ["До якої дати потрібно оплатити гуртожиток за перший семестр?",
              "Яку допомогу отримує студент в академічній відпустці за медичними показаннями?"]:
    show(vector_search, query)
    show(bm25_search, query)

vector_search: До якої дати потрібно оплатити гуртожиток за перший семестр?
  [348] Положення про користування гуртожитками. 5.2. Студент вносить авансовий платіж оплати за 1 семестр (5 місяців)…
  [348] Положення про користування гуртожитками. Мешканці гуртожитків Житомирської політехніки зобов’язані: - знати й …
  [348] Положення про користування гуртожитками. Це є єдиною підставою для поселення і проживання студента у вказаному…

bm25_search: До якої дати потрібно оплатити гуртожиток за перший семестр?
  [348] Положення про користування гуртожитками. 3.9. Здобувачам освіти, які поселяються до гуртожитку Житомирської по…
  [340] Правила призначення академічних стипендій. Цей показник встановлюється у відсотках (у діапазоні від 40 до 45) …
  [340] Правила призначення академічних стипендій. Рішенням ректора Житомирської політехніки таким особам може встанов…

vector_search: Яку допомогу отримує студент в академічній відпустці за медичними показаннями?
  [340] Правила призначення академ

Фрагмент з відповіддю на перше питання містить слова «авансовий платіж оплати… до 1 вересня»,
а не «оплатити»: семантичний пошук знаходить його, а BM25 — лише серед нижчих результатів.
У другому питанні рідкісний вираз «за медичними показаннями» дослівно є у фрагменті з відповіддю,
і BM25 ставить цей фрагмент першим, а семантичний пошук — нижче.

In [8]:
# Експеримент до питання 24: перевірка рангів BM25 vs vector_search на власних питаннях
def find_rank(search_fn, query, doc_id, pattern, k=20):
    for r, doc in enumerate(search_fn(query, k), start=1):
        if doc.metadata["doc_id"] == doc_id and re.search(pattern, doc.page_content, re.IGNORECASE):
            return r
    return None

# 1) Кандидати з точним рідкісним ідентифікатором/власною назвою, де BM25 випереджає vector_search
bm25_candidates = [
    ("Що регулює наказ № 453/од?", "4389", r"453/од"),
    ("Що регулює наказ № 327/од?", "275", r"327/од"),
    ("Хто такий Максим Остапчук?", "277", r"Максим ОСТАПЧУК"),
    ("Що таке форма 063?", "348", r"063"),
]

bm25_winner = None
for q, doc_id, pat in bm25_candidates:
    r_vec = find_rank(vector_search, q, doc_id, pat)
    r_bm25 = find_rank(bm25_search, q, doc_id, pat)
    print(f"[Кандидат BM25] {q} -> vector_search: {r_vec}, bm25_search: {r_bm25}")
    if bm25_winner is None and r_bm25 is not None and (r_vec is None or r_bm25 < r_vec):
        bm25_winner = (q, doc_id, pat, r_vec, r_bm25)

assert bm25_winner is not None, "Жоден кандидат не дав перевагу BM25 — уточніть формулювання запиту."
q_bm25_wins, doc_bm25, pat_bm25, r_vec_1, r_bm25_1 = bm25_winner

# 2) Перефразоване питання синонімами (без точного терміна «академічний плагіат»), де виграє vector_search
q_vec_wins = "Як карають студентів за списування та крадіжку чужих наукових текстів?"
r_vec_2 = find_rank(vector_search, q_vec_wins, "1199", r"5\.3\.2\s+Академічний плагіат")
r_bm25_2 = find_rank(bm25_search, q_vec_wins, "1199", r"5\.3\.2\s+Академічний плагіат")

print("\n=== ПІДСУМОК ПОРІВНЯННЯ ===")
for label, q, rv, rb in [
    ("BM25 випереджає vector_search", q_bm25_wins, r_vec_1, r_bm25_1),
    ("vector_search випереджає BM25", q_vec_wins, r_vec_2, r_bm25_2),
]:
    print(f"[{label}] {q}")
    print(f"  Ранг правильного фрагмента -> vector_search: {rv}, bm25_search: {rb}\n")
    show(vector_search, q, k=3)
    show(bm25_search, q, k=3)

[Кандидат BM25] Що регулює наказ № 453/од? -> vector_search: 10, bm25_search: 1
[Кандидат BM25] Що регулює наказ № 327/од? -> vector_search: 1, bm25_search: 1
[Кандидат BM25] Хто такий Максим Остапчук? -> vector_search: None, bm25_search: 2
[Кандидат BM25] Що таке форма 063? -> vector_search: 6, bm25_search: 1

=== ПІДСУМОК ПОРІВНЯННЯ ===
[BM25 випереджає vector_search] Що регулює наказ № 453/од?
  Ранг правильного фрагмента -> vector_search: 10, bm25_search: 1

vector_search: Що регулює наказ № 453/од?
  [275] Положення про оцінювання результатів навчання (чинна версія 4.0). ЗАТВЕРДЖЕНО Наказ Державного університету «Ж…
  [340] Правила призначення академічних стипендій. IV. Прикінцеві положення 4.1. Правила призначення академічних стипе…
  [275] Положення про оцінювання результатів навчання (чинна версія 4.0). № 327/од Ректор Віктор ЄВДОКИМОВ ПОЛОЖЕННЯ п…

bm25_search: Що регулює наказ № 453/од?
  [4389] Положення про студентського омбудсмана. ЗАТВЕРДЖЕНО Наказ Державного університету

## 5. Гібридний пошук

Два списки результатів обʼєднуємо методом **reciprocal rank fusion** (RRF): фрагмент на позиції $r$
у кожному списку отримує $\frac{1}{60 + r}$ балів, бали з обох списків додаються.
RRF використовує лише позиції, тому не потребує зведення оцінок BM25 і косинусної подібності до однієї шкали.

In [9]:
def rrf(rankings, c=60):
    scores, by_content = {}, {}
    for ranking in rankings:
        for rank, doc in enumerate(ranking, start=1):
            scores[doc.page_content] = scores.get(doc.page_content, 0) + 1 / (c + rank)
            by_content[doc.page_content] = doc
    return [by_content[content] for content in sorted(scores, key=scores.get, reverse=True)]


def hybrid_search(query, k):
    return rrf([vector_search(query, CANDIDATES), bm25_search(query, CANDIDATES)])[:k]


show(hybrid_search, "Якому діапазону балів відповідає оцінка FX?")

hybrid_search: Якому діапазону балів відповідає оцінка FX?
  [3320] ОПП F2 «Інженерія програмного забезпечення» (бакалавр), редакція 2025 року. Оцінювання Оцінювання навчальних д…
  [275] Положення про оцінювання результатів навчання (чинна версія 4.0). Наявні помилки (неточності та ін.) 35 – 59 б…
  [275] Положення про оцінювання результатів навчання (чинна версія 4.0). Під час поточного контролю оцінюванню можуть…



In [10]:
# Обчислення балу RRF вручну (без допоміжних бібліотек) для rank_vec=2, rank_bm25=5, c=60
c = 60
rank_vector = 2
rank_bm25 = 5
rrf_score = 1 / (c + rank_vector) + 1 / (c + rank_bm25)
print(f"RRF score = 1/{c + rank_vector} + 1/{c + rank_bm25} = {rrf_score:.6f}")

RRF score = 1/62 + 1/65 = 0.031514


## 6. Переранжування

Модель ембедингів — **bi-encoder**: запит і фрагмент кодуються окремо, тому ембединги фрагментів можна порахувати
заздалегідь. **Cross-encoder** отримує запит і фрагмент разом і оцінює їхню відповідність точніше,
але для кожної пари потрібен окремий прохід моделі. Тому cross-encoder застосовують лише до кількох десятків
кандидатів, знайдених швидким пошуком.
Використовуємо багатомовну модель [bge-reranker-v2-m3](https://huggingface.co/BAAI/bge-reranker-v2-m3).

In [11]:
reranker = CrossEncoder("BAAI/bge-reranker-v2-m3", device=device, model_kwargs={"dtype": dtype})


def rerank_search(query, k):
    candidates = hybrid_search(query, CANDIDATES)
    scores = reranker.predict([(query, doc.page_content) for doc in candidates])
    return [candidates[i] for i in np.argsort(-scores)[:k]]


show(rerank_search, "Якому діапазону балів відповідає оцінка FX?")

config.json:   0%|          | 0.00/795 [00:00<?, ?B/s]

model.safetensors: reconstructing file:   0%|          |  0.00B / 2.27GB            

model.safetensors: downloading bytes:           |  0.00B            

Loading weights:   0%|          | 0/393 [00:00<?, ?it/s]

tokenizer_config.json:   0%|          | 0.00/1.17k [00:00<?, ?B/s]

sentencepiece.bpe.model: reconstructing file:   0%|          |  0.00B / 5.07MB            

sentencepiece.bpe.model: downloading bytes:           |  0.00B            

tokenizer.json: reconstructing file:   0%|          |  0.00B / 17.1MB            

tokenizer.json: downloading bytes:           |  0.00B            

special_tokens_map.json:   0%|          | 0.00/964 [00:00<?, ?B/s]

rerank_search: Якому діапазону балів відповідає оцінка FX?
  [275] Положення про оцінювання результатів навчання (чинна версія 4.0). Наявні помилки (неточності та ін.) 35 – 59 б…
  [3320] ОПП F2 «Інженерія програмного забезпечення» (бакалавр), редакція 2025 року. Оцінювання Оцінювання навчальних д…
  [275] Положення про оцінювання результатів навчання (чинна версія 4.0). Значення вагового коефіцієнта для кожного ви…



## 7. Бенчмарк

Набір питань складено **до** налаштування системи. Для кожного питання вказано документи, у яких є відповідь,
регулярний вираз, за яким можна впізнати фрагмент із відповіддю, та еталонну відповідь. Три питання не мають відповіді в документах: на них система має відмовитися відповідати.

In [12]:
BENCHMARK = [
    ("До якої дати потрібно оплатити гуртожиток за перший семестр?", ["348"], r"1 вересня", "До 1 вересня поточного року."),
    ("До якої дати оплачують гуртожиток за другий семестр?", ["348"], r"1 лютого", "До 1 лютого поточного року."),
    ("З якої до якої години в гуртожитку має бути тиша?", ["1219"], r"23:00 години і до 7:00", "З 23:00 до 7:00."),
    ("У який час гості можуть перебувати в гуртожитку?", ["1219", "348"], r"0?8:00 до 22:00", "З 8:00 до 22:00."),
    ("Чи можна тримати в гуртожитку домашню тварину?", ["1219", "348"], r"утримувати тварин", "Ні, утримувати тварин у гуртожитку заборонено."),
    ("На який строк укладають договір найму житла в гуртожитку?", ["348"], r"терміном на один навчальний рік", "На один навчальний рік."),
    ("За який час потрібно звільнити гуртожиток після відрахування?", ["1219"], r"протягом двох тижнів", "Протягом двох тижнів від дня виходу наказу."),
    ("Коли обирають старосту на першому курсі?", ["336"], r"після двох тижнів", "На початку навчального року, після двох тижнів навчального процесу."),
    ("На який строк обирається староста групи?", ["336"], r"на весь період навчання", "На весь період навчання."),
    ("Скільки балів потрібно для оцінки «добре» B?", ["275"], r"82 ?– ?89", "82–89 балів."),
    ("Якому діапазону балів відповідає оцінка FX?", ["275"], r"35 ?– ?59", "35–59 балів."),
    ("Яку частку кредитів становлять вибіркові дисципліни?", ["277"], r"25 відсотків",
     "Не менше 25 відсотків загальної кількості кредитів ЄКТС (не менше 10 відсотків для спеціальностей з додатковим регулюванням)."),
    ("До якої дати викладачі вносять анотації вибіркових дисциплін?", ["277"], r"30 квітня", "До 30 квітня."),
    ("Як часто виплачують стипендію?", ["340"], r"один раз на місяць", "Один раз на місяць."),
    ("За яким показником формують рейтинг для призначення академічної стипендії?", ["340"], r"середнім балом", "За середнім балом семестрового контролю."),
    ("Яку допомогу отримує студент-бюджетник в академічній відпустці за медичними показаннями?", ["340"], r"50 відсотків мінімальної",
     "Щомісячну допомогу в розмірі 50 відсотків мінімальної академічної стипендії."),
    ("Для кого встановлюється стипендія ректора?", ["344"], r"встановлюється для студентів денної",
     "Для студентів денної форми навчання із суттєвими досягненнями в навчанні, науковій, творчій і громадській діяльності."),
    ("На який строк обирають студентського омбудсмана?", ["4389"], r"терміном на 1 рік", "На 1 рік."),
    ("До якої дати проводять вибори студентського омбудсмана?", ["4389"], r"до 01 жовтня", "До 1 жовтня кожного навчального року."),
    ("Що таке самоплагіат?", ["1199"], r"Самоплагіат –", "Оприлюднення власних раніше опублікованих результатів без зазначення джерела попереднього оприлюднення."),
    ("Що таке фабрикація в академічній діяльності?", ["1199"], r"Фабрикація –", "Вигадування даних чи фактів про результати власної академічної діяльності."),
    ("Який обсяг бакалаврської програми «Інженерія програмного забезпечення» в кредитах ЄКТС?", ["3320", "3329"], r"240 кредит", "240 кредитів ЄКТС."),
    ("Який шифр спеціальності «Інженерія програмного забезпечення» в чинній освітній програмі?", ["3320"], r"спеціальн\w* F2", "F2."),
    ("Скільки коштує обід у студентській їдальні?", [], None, None),
    ("Який графік роботи бібліотеки університету?", [], None, None),
    ("Скільки коштує рік навчання на контракті за спеціальністю «Інженерія програмного забезпечення»?", [], None, None),
]
answerable = [(q, gold, evidence) for q, gold, evidence, _ in BENCHMARK if gold]
print(f"Питань: {len(BENCHMARK)}, з них без відповіді в документах: {len(BENCHMARK) - len(answerable)}")

Питань: 26, з них без відповіді в документах: 3


### Метрики ретриву

Фрагмент вважаємо правильним, якщо він походить із правильного документа і містить вираз-доказ.
Перевіряти лише документ замало: у довгому документі десятки фрагментів, і більшість із них відповіді не містять.

- **hit rate@k** — частка питань, для яких правильний фрагмент є серед `k` знайдених;
- **MRR** (mean reciprocal rank) — середнє за питаннями значення $\frac{1}{r}$, де $r$ — позиція першого
  правильного фрагмента; якщо його немає серед перших 10, замість $\frac{1}{r}$ береться 0. MRR враховує не лише те, чи знайдено фрагмент, а й те, наскільки високо.

Питання без відповіді в ретриві не оцінюємо: правильного фрагмента для них немає.

In [13]:
def first_hit_rank(results, gold, evidence):
    for rank, doc in enumerate(results, start=1):
        if doc.metadata["doc_id"] in gold and re.search(evidence, doc.page_content):
            return rank
    return None


def hit_rate(ranks, k):
    return np.mean([rank is not None and rank <= k for rank in ranks])


def mrr(ranks):
    return np.mean([1 / rank if rank else 0 for rank in ranks])


def retrieval_metrics(search):
    ranks = [first_hit_rank(search(question, 10), gold, evidence) for question, gold, evidence in answerable]
    return {"метод": search.__name__, "hit@1": hit_rate(ranks, 1), "hit@5": hit_rate(ranks, 5), "MRR": mrr(ranks)}


pd.DataFrame([retrieval_metrics(search) for search in [vector_search, bm25_search, hybrid_search, rerank_search]]).round(2)

,метод,hit@1,hit@5,MRR
0,vector_search,0.61,0.87,0.74
1,bm25_search,0.52,0.65,0.59
2,hybrid_search,0.61,0.91,0.76
3,rerank_search,0.83,0.96,0.90


На українських текстах BM25 сам по собі зазвичай поступається семантичному пошуку: слова мають багато форм,
а питання рідко повторюють формулювання документа. Переранжування найсильніше впливає на hit@1:
cross-encoder бачить запит і фрагмент разом і частіше ставить правильний фрагмент на перше місце.
Висновки варто робити обережно: на 23 питаннях одне питання — це понад 4 відсоткові пункти.

In [14]:
# Експеримент до питання 27: додаємо 3 власні питання до BENCHMARK
extended_benchmark = BENCHMARK + [
    ("Як відбувається звільнення старости групи від виконання його обовʼязків?", ["336"], r"Звільнення від виконання обов",
     "Звільнення від виконання обовʼязків старости відбувається за ініціативою деканату або студентів групи."),
    ("До якого числа першокурсники мають обрати старосту?", ["336"], r"після двох тижнів",
     "На початку навчального року, після двох тижнів навчального процесу."),  # перефразування
    ("Скільки коштує абонемент у басейн університету?", [], None, None),     # без відповіді
]

# Перевірка, що кожен новий regex дійсно є у відповідному документі
for q, gold, ev, _ in extended_benchmark[-3:]:
    if gold:
        matched = any(c.metadata["doc_id"] in gold and re.search(ev, c.page_content) for c in chunks)
        assert matched, f"Regex {ev!r} не знайдено у документах {gold}!"

answerable_ext = [(q, gold, evidence) for q, gold, evidence, _ in extended_benchmark if gold]

def retrieval_metrics_ext(search):
    ranks = [first_hit_rank(search(q, 10), gold, ev) for q, gold, ev in answerable_ext]
    return {"метод": search.__name__, "hit@1": hit_rate(ranks, 1), "hit@5": hit_rate(ranks, 5), "MRR": mrr(ranks)}

display(pd.DataFrame([retrieval_metrics_ext(s) for s in [vector_search, bm25_search, hybrid_search, rerank_search]]).round(2))

,метод,hit@1,hit@5,MRR
0,vector_search,0.64,0.88,0.76
1,bm25_search,0.52,0.64,0.58
2,hybrid_search,0.64,0.92,0.78
3,rerank_search,0.84,0.96,0.90


## 8. RAG-ланцюжок на LangChain

Генеративна модель та сама, що в демо 2. Загортаємо її в `HuggingFacePipeline`, а `ChatHuggingFace`
застосовує до повідомлень шаблон чату моделі.

In [15]:
GENERATOR_MODEL_NAME = "Qwen/Qwen3-4B-Instruct-2507"
tokenizer = AutoTokenizer.from_pretrained(GENERATOR_MODEL_NAME)
llm = AutoModelForCausalLM.from_pretrained(GENERATOR_MODEL_NAME, dtype=dtype).to(device)

generator = pipeline(
    "text-generation", model=llm, tokenizer=tokenizer,
    max_new_tokens=300, do_sample=False, temperature=None, top_p=None, top_k=None, return_full_text=False,
)
chat = ChatHuggingFace(llm=HuggingFacePipeline(pipeline=generator), tokenizer=tokenizer)

config.json:   0%|          | 0.00/727 [00:00<?, ?B/s]

tokenizer_config.json:   0%|          | 0.00/9.38k [00:00<?, ?B/s]

vocab.json:   0%|          | 0.00/2.78M [00:00<?, ?B/s]

merges.txt:   0%|          | 0.00/1.67M [00:00<?, ?B/s]

tokenizer.json: reconstructing file:   0%|          |  0.00B / 11.4MB            

tokenizer.json: downloading bytes:           |  0.00B            

model.safetensors.index.json:   0%|          | 0.00/32.8k [00:00<?, ?B/s]

Reconstructing (incomplete total...): |          |  0.00B /  0.00B            

Fetching 3 files:   0%|          | 0/3 [00:00<?, ?it/s]

Loading weights:   0%|          | 0/398 [00:00<?, ?it/s]

generation_config.json:   0%|          | 0.00/238 [00:00<?, ?B/s]

Ланцюжок записуємо мовою LCEL (LangChain Expression Language): оператор `|` передає вихід одного кроку
на вхід наступного. Ланцюжок повертає і відповідь, і знайдені фрагменти: вони потрібні для перевірки цитувань.

In [16]:
SYSTEM_PROMPT = (
    "Ти — асистент, що відповідає на питання студентів Житомирської політехніки. "
    "Відповідай лише на основі наданих фрагментів документів. "
    "Після кожного твердження вказуй у квадратних дужках номер фрагмента, на якому воно ґрунтується, наприклад [2]. "
    "Якщо у фрагментах немає відповіді на питання, відповідай лише: «У наданих документах немає відповіді.» "
    "Відповідай українською мовою, коротко."
)
prompt = ChatPromptTemplate.from_messages([
    ("system", SYSTEM_PROMPT),
    ("human", "Фрагменти документів:\n\n{context}\n\nПитання: {question}"),
])


def format_docs(docs):
    return "\n\n".join(f"[{n}] {doc.page_content}" for n, doc in enumerate(docs, start=1))


retriever = RunnableLambda(lambda question: rerank_search(question, 5))
answer_chain = (
    RunnableLambda(lambda x: {"context": format_docs(x["docs"]), "question": x["question"]})
    | prompt
    | chat
    | StrOutputParser()
)
rag_chain = {"docs": retriever, "question": RunnablePassthrough()} | RunnablePassthrough.assign(answer=answer_chain)

result = rag_chain.invoke("Чи можна тримати в гуртожитку домашню тварину?")
print(result["answer"])
for n, doc in enumerate(result["docs"], start=1):
    print(f"  [{n}] {doc.metadata['doc_id']}: {doc.metadata['title']}")

Ні, в гуртожитку домашню тварину тримати заборонено [1].
  [1] 348: Положення про користування гуртожитками
  [2] 348: Положення про користування гуртожитками
  [3] 348: Положення про користування гуртожитками
  [4] 1219: Правила внутрішнього розпорядку в студентських гуртожитках
  [5] 1219: Правила внутрішнього розпорядку в студентських гуртожитках


## 9. Оцінювання відповідей

Для кожного питання бенчмарку перевіряємо:
- **правильність**: для питань із відповіддю — чи збігається відповідь з еталонною; для питань без відповіді — чи відмовилася система;
- **цитування**: чи є серед фрагментів, на які посилається відповідь, фрагмент із правильного документа.

Правильність вільної відповіді важко перевірити правилом: «до 1 вересня» і «не пізніше першого вересня» — одна відповідь.
Тому використовуємо підхід **LLM як суддя**: інша (тут — та сама) модель порівнює відповідь з еталоном.
Суддя теж помиляється, тому його вердикти варто вибірково перевіряти вручну.

In [17]:
REFUSAL = "немає відповіді"
JUDGE_PROMPT = (
    "Порівняй відповідь системи з еталонною відповіддю на питання.\n\n"
    "Питання: {question}\nЕталонна відповідь: {reference}\nВідповідь системи: {answer}\n\n"
    "Чи містить відповідь системи ту саму інформацію, що й еталонна, без суперечностей? "
    "Відповідай одним словом: «так» або «ні»."
)


def judge(question, reference, answer):
    verdict = chat.invoke(JUDGE_PROMPT.format(question=question, reference=reference, answer=answer)).content
    return verdict.strip().lower().startswith("так")


def cited_doc_ids(answer, retrieved):
    # Номери на зразок [2] у відповіді перетворюємо на ідентифікатори документів
    numbers = [int(n) for n in re.findall(r"\[(\d+)\]", answer)]
    return [retrieved[n - 1] for n in numbers if 0 < n <= len(retrieved)]


def evaluate(question, gold, reference):
    result = rag_chain.invoke(question)
    answer = result["answer"]
    retrieved = [doc.metadata["doc_id"] for doc in result["docs"]]
    if gold:
        correct = judge(question, reference, answer)
        citation_ok = any(doc_id in gold for doc_id in cited_doc_ids(answer, retrieved))
    else:
        correct = REFUSAL in answer.lower()
        citation_ok = None
    return {"питання": question, "правильно": correct, "цитування": citation_ok,
            "документи": gold, "знайдено": retrieved, "відповідь": answer}


evaluation = pd.DataFrame([evaluate(question, gold, reference) for question, gold, _, reference in BENCHMARK])
has_answer = evaluation["цитування"].notna()
print(f"Правильних відповідей на питання з відповіддю: {evaluation[has_answer]['правильно'].mean():.0%}")
print(f"Коректних цитувань: {evaluation[has_answer]['цитування'].astype(bool).mean():.0%}")
print(f"Правильних відмов на питання без відповіді: {evaluation[~has_answer]['правильно'].mean():.0%}")

[transformers] You seem to be using the pipelines sequentially on GPU. In order to maximize efficiency please use a dataset


Правильних відповідей на питання з відповіддю: 91%
Коректних цитувань: 96%
Правильних відмов на питання без відповіді: 100%


In [18]:
# Виведення 5 відповідей (3 правильні + 2 помилкові) для ручної перевірки в питанні 29
ref_map = {q: ref for q, _, _, ref in BENCHMARK}
sample_indices = [0, 3, 10, 16, 22]  # 0, 3, 10 — з відповіддю (True); 16, 22 — вердикт судді False
for idx in sample_indices:
    row = evaluation.iloc[idx]
    print(f"[{idx}] Питання: {row['питання']}")
    print(f"    Еталон:    {ref_map[row['питання']]}")
    print(f"    Відповідь: {row['відповідь']}")
    print(f"    Вердикт судді: {row['правильно']} | Цитування: {row['цитування']}\n")

[0] Питання: До якої дати потрібно оплатити гуртожиток за перший семестр?
    Еталон:    До 1 вересня поточного року.
    Відповідь: До 1 вересня поточного року потрібно оплатити гуртожиток за перший семестр [1].
    Вердикт судді: True | Цитування: True

[3] Питання: У який час гості можуть перебувати в гуртожитку?
    Еталон:    З 8:00 до 22:00.
    Відповідь: Гості можуть перебувати в гуртожитку з 08:00 до 22:00 години [1].
    Вердикт судді: True | Цитування: True

[10] Питання: Якому діапазону балів відповідає оцінка FX?
    Еталон:    35–59 балів.
    Відповідь: Оцінка FX відповідає діапазону балів 35 – 59. [1]
    Вердикт судді: True | Цитування: True

[16] Питання: Для кого встановлюється стипендія ректора?
    Еталон:    Для студентів денної форми навчання із суттєвими досягненнями в навчанні, науковій, творчій і громадській діяльності.
    Відповідь: Для студентів денної форми навчання, які мають суттєві досягнення у навчанні, науковій, творчій та громадській діяльності, є пе

## 10. Аналіз відмов

Найкорисніша частина оцінювання — не підсумкове число, а розбір помилок. Для кожної неправильної відповіді
дивимося, чи був серед знайдених фрагментів правильний документ:
- якщо **не був** — це промах ретриву, і виправляти треба пошук (розбиття, модель ембедингів, гібридний пошук);
- якщо **був** — модель не скористалася контекстом або скористалася неправильно, і виправляти треба промпт чи модель.

In [19]:
def retrieval_verdict(row):
    if not row["документи"] or any(doc_id in row["документи"] for doc_id in row["знайдено"]):
        return "ретрив знайшов"
    return "промах ретриву"


def describe_failure(row):
    print(f"Питання: {row['питання']}")
    print(f"  правильний документ: {row['документи']}, знайдено: {row['знайдено']} — {retrieval_verdict(row)}")
    print(f"  правильно: {row['правильно']}, цитування: {row['цитування']}")
    print(f"  відповідь: {row['відповідь']}\n")


failures = evaluation[(~evaluation["правильно"]) | (evaluation["цитування"] == False)]
for _, row in failures.iterrows():
    describe_failure(row)

Питання: Для кого встановлюється стипендія ректора?
  правильний документ: ['344'], знайдено: ['344', '340', '344', '340', '340'] — ретрив знайшов
  правильно: False, цитування: True
  відповідь: Для студентів денної форми навчання, які мають суттєві досягнення у навчанні, науковій, творчій та громадській діяльності, є переможцями міжнародних та всеукраїнських олімпіад, конкурсів та змагань [1].

Питання: Який шифр спеціальності «Інженерія програмного забезпечення» в чинній освітній програмі?
  правильний документ: ['3320'], знайдено: ['3329', '3329', '3320', '3329', '3329'] — ретрив знайшов
  правильно: False, цитування: False
  відповідь: Шифр спеціальності «Інженерія програмного забезпечення» в чинній освітній програмі — 121 [2].



Кожну неправильну відповідь відносять до однієї з двох груп:
- **помилка оцінювання**: відповідь правильна, але суддя помилився (відповідь сформульована інакше або містить більше деталей, ніж еталон)
  або еталон неоднозначний (відповідь спирається на інше місце документа, ніж еталон, і не суперечить документу);
- **помилка системи**: промах ретриву, галюцинація попри наявний контекст або суперечливі джерела —
  як у питанні про шифр спеціальності, де чинна редакція програми сама містить старий шифр 121 (див. демо 2).

Перед тим як змінювати систему, варто перевірити, чи правильно оцінено відповідь.

## Підсумок

- LangChain дає готові компоненти — розбивачі тексту, обгортки моделей, векторні сховища, ретривери — і спосіб
  зʼєднати їх у ланцюжок; логіку пошуку, як-от гібридний пошук чи переранжування, легко додати власними функціями.
- BM25 і семантичний пошук помиляються на різних питаннях, тому їхнє поєднання зазвичай краще за кожен окремо.
- Cross-encoder точніше оцінює відповідність фрагмента запиту, але працює лише на невеликій кількості кандидатів.
- Ретрив і генерацію оцінюють окремо: так видно, де саме виникає помилка.
- Для автоматичного оцінювання відповідей існують готові бібліотеки, наприклад [RAGAS](https://docs.ragas.io/);
  вони теж спираються на LLM як суддю.

## Відповіді на контрольні питання (21–30)

21. **Тривалість виконання `03_rag_langchain_eval` та проблеми запуску:**  
    Усі клітинки ноутбука на GPU (`cuda`) виконано приблизно за **5–7 хвилин** (основний час зайняло оцінювання 26 питань бенчмарку з реранкером `bge-reranker-v2-m3`, генерацією відповідей і повторними викликами LLM-судді). Критичних проблем із запуском не було; спостерігалися лише некритичні попередження: конфлікт версій `requests` у середовищі Colab (`pip`), `DeprecationWarning` для `langchain-community`, попередження про відсутність `HF_TOKEN` та рекомендація `transformers` передавати батчі замість послідовного виклику `pipeline`.

22. **Відмінність `RecursiveCharacterTextSplitter` від розбиття за реченнями (демо 2) та роль `keep_separator="end"`:**  
    - Самописний розбивач із демо 2 ділить текст одним регулярним виразом по кінцях речень і пакує їх із перекриттям в одне останнє речення (дав **410** фрагментів). `RecursiveCharacterTextSplitter` працює **ієрархічно**: спершу намагається розбити текст за першим роздільником зі списку `separators=[". ", "; ", " ", ""]`, а якщо фрагмент досі перевищує `chunk_size=1000`, рекурсивно дробить його за наступним роздільником із фіксованим перекриттям `chunk_overlap=200` (дав **385** фрагментів).  
    - Параметр `keep_separator="end"` залишає роздільник (наприклад, крапку `". "`) у кінці попереднього фрагмента, а не переносить його на початок наступного, зберігаючи природну пунктуацію тексту.

23. **Чому для FAISS обрано `DistanceStrategy.MAX_INNER_PRODUCT` і чи можна було б без нормалізації:**  
    Оскільки `HuggingFaceEmbeddings` повертає $L_2$-нормалізовані вектори ($\|\mathbf{a}\| = \|\mathbf{b}\| = 1$), скалярний добуток (`MAX_INNER_PRODUCT`) математично тотожний косинусній подібності, але швидший в обчисленні. Без нормалізації використовувати `MAX_INNER_PRODUCT` замість косинусної подібності було б некоректно: довші за евклідовою нормою вектори отримували б вищі оцінки навіть за меншої семантичної близькості (більшого кута між векторами).

24. **Порівняння BM25 та семантичного пошуку на власних питаннях (розділ 4):**  
    - **Питання, де виграє BM25:** *«Що регулює наказ № 453/од?»* (документ `4389`) — **`bm25_search`** поставив правильний фрагмент на **1-шу позицію**, тоді як **`vector_search`** знайшов його лише на **10-й позиції** (піднявши в топ-3 накази про оцінювання `275` і стипендії `340`). Перевагу BM25 підтвердили й інші перевірені запити з точними ідентифікаторами без слів-підказок: *«Що таке форма 063?»* (`bm25_search: 1` проти `vector_search: 6`) і *«Хто такий Максим Остапчук?»* (`bm25_search: 2` проти `vector_search: None`). Це відбувається тому, що BM25 надає високу IDF-вагу точним рідкісним токенам (`453`, `063`, `максим`, `остапчук`), тоді як семанчний пошук без контекстних слів («омбудсман», «гуртожиток») орієнтується на загальний зміст («наказ», «що регулює»).  
    - **Питання, де виграє семантичний пошук:** *«Як карають студентів за списування та крадіжку чужих наукових текстів?»* (документ `1199`) — **`vector_search`** вивів пункт `5.3.2 Академічний плагіат` на **2-гу позицію**, розпізнавши синоніми «списування / крадіжка чужих текстів» ↔ «академічний плагіат», тоді як **`bm25_search`** поставив його лише на **15-ту позицію** (поза топ-3), орієнтуючись на часті слова («студентів», «наукових», «текстів»).

25. **Ручне обчислення балу RRF (розділ 5):**  
    - За формулою $\text{RRF}(d) = \sum_i \frac{1}{60 + r_i}$ при позиції $r_{\text{vec}} = 2$ у векторному пошуку та $r_{\text{bm25}} = 5$ у BM25 (реалізовано чистим Python у клітинці 10):  
      $\displaystyle \text{RRF} = \frac{1}{60 + 2} + \frac{1}{60 + 5} = \frac{1}{62} + \frac{1}{65} \approx 0.016129 + 0.015385 = \mathbf{0.031514}$  
    - RRF оперує лише рангами (`1, 2, 3, …`), бо оцінки BM25 (необмежені додатні значення, залежні від частоти слів і довжини документа) та косинусна подібність (діапазон $[-1, 1]$) мають різні шкали й розподіли, тож їх не можна коректно складати напряму без складної нормалізації.

26. **Чому cross-encoder застосовують лише до 20 кандидатів і скільки проходів потрібно на весь корпус (розділ 6):**  
    Bi-encoder кодує всі фрагменти один раз під час індексації, а під час запиту обчислює лише ембединг питання й матричний добуток. Cross-encoder (`BAAI/bge-reranker-v2-m3`) приймає пару `(запит, фрагмент)` разом через механізм повної уваги (cross-attention), тому попередньо обчислити вектори неможливо. Щоб переранжувати весь корпус на одне питання, знадобилося б **385 проходів моделі** (за кількістю `len(chunks) = 385`), що майже у **19.25 раза** повільніше, ніж оцінити 20 кандидатів.

27. **Додавання 3 власних питань до бенчмарку (розділ 7):**  
    Додано 3 питання (1 з відповіддю, 1 перефразування наявного питання про обрання старости, 1 без відповіді про басейн — разом 29 питань, із них 25 із відповіддю та 4 без відповіді).  
    - На початковому наборі (23 питання з відповіддю): `vector_search` — hit@1 `0.61`, hit@5 `0.87`, MRR `0.74`; `bm25_search` — `0.52` / `0.65` / `0.59`; `hybrid_search` — `0.61` / `0.91` / `0.76`; `rerank_search` — `0.83` / `0.96` / `0.90`.  
    - Після додавання 3 питань (25 питань із відповіддю): `vector_search` — `hit@1 = 0.64` (+0.03), `hit@5 = 0.88` (+0.01), `MRR = 0.76` (+0.02); `bm25_search` — `hit@1 = 0.52` (0.00), `hit@5 = 0.64` (-0.01), `MRR = 0.58` (-0.01); `hybrid_search` — `hit@1 = 0.64` (+0.03), `hit@5 = 0.92` (+0.01), `MRR = 0.78` (+0.02); `rerank_search` — `hit@1 = 0.84` (+0.01), `hit@5 = 0.96` (0.00), `MRR = 0.90` (0.00).  
    - Для перефразованого питання (*«До якого числа першокурсники мають обрати старосту?»*) семантичний і гібридний пошук упевнено знайшли потрібний фрагмент, що злегка підняло їхні метрики, тоді як BM25 через інше слововживання («першокурсники мають обрати» замість «вибори старости на 1 курсі») дещо знизив `hit@5` та `MRR`. Питання без відповіді (*«Скільки коштує абонемент у басейн університету?»*) на метрики ретриву не впливає, бо ретрив оцінюється лише на підмножині `answerable`.

28. **Порівняння чотирьох методів пошуку (розділ 7) та вплив переранжування на `hit@1`:**  
    За результатами бенчмарку (23 питання з відповіддю):
    - `bm25_search`: `hit@1 = 0.52`, `hit@5 = 0.65`, `MRR = 0.59` — найнижчі показники через багату словозміну української мови та відмінні формулювання в питаннях.
    - `vector_search`: `hit@1 = 0.61`, `hit@5 = 0.87`, `MRR = 0.74` — значно краще знаходить синоніми й перефразування.
    - `hybrid_search`: `hit@1 = 0.61`, `hit@5 = 0.91`, `MRR = 0.76` — поєднання з BM25 підняло повноту `hit@5` (+4 в.п.) і `MRR` (+2 в.п.).
    - `rerank_search`: `hit@1 = 0.83`, `hit@5 = 0.96`, `MRR = 0.90` — найкращі метрики.  
    Переранжування найсильніше підняло саме **`hit@1`** (стрибок із `0.61` до `0.83`, тобто **+22 в.п.**), оскільки cross-encoder бачить усі токени запиту й кандидата одночасно, розрізняє конкретні умови, числа та заперечення й виводить найточніший фрагмент із 2–5 позицій на перше місце.

29. **Проблеми використання тієї самої моделі як судді (розділ 9) та ручна перевірка 5 відповідей:**  
    - **Ризики:** упередження до власного стилю формулювань (self-preference bias), дублювання помилок інтерпретації контексту (суддя має ті самі обмеження, що й генератор) та надмірна прискіпливість до розгорнутих правильних відповідей при порівнянні зі стислим еталоном.  
    - **Ручна перевірка 5 відповідей (з виводу додаткової клітинки після розділу 9):**  
      1. `[0]` *«До якої дати потрібно оплатити гуртожиток за перший семестр?»* — еталон: *«До 1 вересня поточного року.»*, відповідь: *«До 1 вересня поточного року потрібно оплатити гуртожиток за перший семестр [1].»* — вердикт судді: **так (`True`)**, моя перевірка: **збігається** (факт і цитування `[1]` точні).  
      2. `[3]` *«У який час гості можуть перебувати в гуртожитку?»* — еталон: *«З 8:00 до 22:00.»*, відповідь: *«Гості можуть перебувати в гуртожитку з 08:00 до 22:00 години [1].»* — вердикт судді: **так (`True`)**, моя перевірка: **збігається**.  
      3. `[10]` *«Якому діапазону балів відповідає оцінка FX?»* — еталон: *«35–59 балів.»*, відповідь: *«Оцінка FX відповідає діапазону балів 35 – 59. [1]»* — вердикт судді: **так (`True`)**, моя перевірка: **збігається**.  
      4. `[16]` *«Для кого встановлюється стипендія ректора?»* — відповідь повністю відтворює еталон і додає з того ж пункту документа `344` уточнення про переможців олімпіад, конкурсів та змагань `[1]` — вердикт судді: **ні (`False`)**, моя перевірка: **розбігається (помилка судді)**.  
      5. `[22]` *«Який шифр спеціальності «Інженерія програмного забезпечення» в чинній освітній програмі?»* — еталон: *«F2.»*, відповідь: *«... — 121 [2].»* — вердикт судді: **ні (`False`)**, моя перевірка: **збігається** (система назвала старий шифр `121` і послалася на застарілий документ `3329`).

30. **Аналіз неправильних відповідей у розділі 10 «Аналіз відмов»:**  
    На бенчмарку з 26 питань отримано `91%` правильних відповідей (21/23), `96%` коректних цитувань (22/23) та `100%` правильних відмов (3/3). У `failures` потрапили **2 випадки**:  
    1. **«Для кого встановлюється стипендія ректора?»** (`правильний документ: ['344']`, `знайдено: ['344', '340', '344', '340', '340']`, `правильно: False`, `цитування: True`).  
       - **Тип помилки:** **помилка судді** (помилка оцінювання).  
       - **Обґрунтування:** ретрив поставив правильний документ `344` на 1-ше місце, генератор надав точну відповідь із коректним цитуванням `[1]`, яка повністю відповідає еталону. Суддя помилково поставив `False` лише тому, що модель додала з того самого пункту документа уточнення про переможців міжнародних і всеукраїнських олімпіад, конкурсів та змагань.  
    2. **«Який шифр спеціальності «Інженерія програмного забезпечення» в чинній освітній програмі?»** (`правильний документ: ['3320']`, `знайдено: ['3329', '3329', '3320', '3329', '3329']`, `правильно: False`, `цитування: False`).  
       - **Тип помилки:** **помилка генерації (і вибору контексту), спричинена застарілими й суперечливими джерелами без фільтрації метаданих** (серед 4 категорій запитання це не промах ретриву — бо `3320` був у топ-5 на позиції `[3]`, — не помилка судді і не неоднозначний еталон, а помилка системи/генерації через домінування застарілої редакції).  
       - **Обґрунтування:** у демо 3 застаріла редакція ОПП 2024 року (`3329`) не відфільтрована, тому вона зайняла 4 із 5 позицій у топі (`[1], [2], [4], [5]`), а модель сперлася на фрагмент `[2]` (`3329`) і видала старий шифр `121` замість `F2` (що також призвело до хибного цитування). Додатково в самому чинному документі `3320` старий шифр `121` трапляється частіше за `F2` (5 проти 2 фрагментів).